# SAP Prediction — Confusion Matrices

This notebook visualises how the PenuX-AP-Severity model performs across different decision thresholds.

**Three views:**
1. Single confusion matrix at the default clinical threshold
2. Grid of matrices across the full threshold sweep
3. Metrics-vs-threshold curve (Sensitivity, Specificity, PPV, NPV, F1)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import matplotlib.pyplot as plt
from penux_ap.evaluation import (
    evaluate_binary_classifier,
    plot_confusion_matrix,
    plot_confusion_matrix_sweep,
    plot_metrics_vs_threshold,
    confusion_matrix_at_thresholds,
    threshold_table,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## Load model predictions

Replace the cell below with your actual `y_true` / `y_proba` arrays,
or load them from a saved evaluation run.

In [ ]:
# ── Option A: load from a saved evaluation ──────────────────────────────────
# import json, numpy as np
# with open('../outputs/eval/eval_metrics.json') as f:
#     saved = json.load(f)

# ── Option B: generate synthetic demo data (REMOVE for real use) ─────────────
rng = np.random.default_rng(42)
n = 400
y_true  = rng.integers(0, 2, size=n)                          # true SAP labels
y_proba = np.clip(
    y_true * 0.55 + rng.normal(0, 0.22, size=n), 0, 1        # simulated scores
)
print(f"Cohort: {n} patients  |  SAP rate: {y_true.mean():.1%}")

## 1 — Single confusion matrix at default threshold (0.40)

In [ ]:
THRESHOLD = 0.40

metrics = evaluate_binary_classifier(y_true, y_proba, threshold=THRESHOLD)
print(f"AUROC      : {metrics['auroc']:.3f}")
print(f"Sensitivity: {metrics['sensitivity']:.3f}")
print(f"Specificity: {metrics['specificity']:.3f}")
print(f"PPV        : {metrics['ppv']:.3f}")
print(f"NPV        : {metrics['npv']:.3f}")
print(f"F1         : {metrics['f1']:.3f}")

fig, ax = plt.subplots(figsize=(5, 4))
plot_confusion_matrix(y_true, y_proba, threshold=THRESHOLD, ax=ax)
plt.tight_layout()
plt.show()

## 2 — Grid: confusion matrices across 7 thresholds

In [ ]:
fig = plot_confusion_matrix_sweep(
    y_true, y_proba,
    thresholds=[0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80],
    save_path='../outputs/reports/confusion_matrix_sweep.png',
)
plt.show()

## 3 — Metrics vs. threshold curve

In [ ]:
fig = plot_metrics_vs_threshold(
    y_true, y_proba,
    highlight_threshold=THRESHOLD,
    save_path='../outputs/reports/metrics_vs_threshold.png',
)
plt.show()

## 4 — Full threshold table (numeric)

In [ ]:
df = threshold_table(y_true, y_proba)
df.style.format({
    'threshold':   '{:.2f}',
    'sensitivity': '{:.3f}',
    'specificity': '{:.3f}',
    'ppv':         '{:.3f}',
    'npv':         '{:.3f}',
    'f1':          '{:.3f}',
    'accuracy':    '{:.3f}',
}).background_gradient(subset=['sensitivity', 'specificity', 'f1'], cmap='RdYlGn')

## 5 — Compare to BISAP / APACHE-II / Ranson baselines

If you have external score predictions, compare them here:

In [ ]:
# Example: if you have BISAP predictions
# bisap_proba = ...
# ranson_proba = ...

# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# plot_confusion_matrix(y_true, y_proba,     threshold=0.40, ax=axes[0], title='PenuX-AP')
# plot_confusion_matrix(y_true, bisap_proba,  threshold=0.40, ax=axes[1], title='BISAP')
# plot_confusion_matrix(y_true, ranson_proba, threshold=0.40, ax=axes[2], title='Ranson')
# plt.tight_layout()
# plt.show()

print("Uncomment the cells above and supply external score arrays to compare.")